In [8]:
#TASK1 

import pandas as pd
import plotly.express as px


# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

# 1. Filter to fossil sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# 2. Compute insight: identify the region with the highest total fossil TWh
region_fossil_totals = df_fossil.groupby('Region')['TWh'].sum()
top_region = region_fossil_totals.idxmax()
top_region_val = region_fossil_totals.max()

# 3. Define a CVD-safe color palette for the fossil sources (Coal, Oil, Natural Gas)
# Using ColorBrewer/Okabe-Ito inspired CVD-safe hex codes
cvd_fossil_colors = {
    'Coal': '#0072B2',        # Blue
    'Natural Gas': '#E69F00', # Orange
    'Oil': '#CC79A7'          # Reddish-purple / Pink
}

# 4. Build the Treemap
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=cvd_fossil_colors,
    title=f"<b>Fossil Fuel Dependency by Region, Country, & Source</b><br><sup>Insight: {top_region} leads global fossil consumption ({top_region_val:,.0f} TWh)</sup>"
)

# 5. Customize layout and traces
fig.update_traces(
    textinfo="label+value",          # Show label and absolute TWh values, omit percentages
    root_color="#E0E0E0"             # Grey out parent/root nodes
)

fig.update_layout(
    margin=dict(t=60, l=25, r=25, b=25),
    font=dict(size=12)
)

fig.show()

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


In [9]:
#TASK2


import pandas as pd
import plotly.express as px

# 1. Filter to Low-carbon sources and aggregate TWh by country
df_low_carbon = df.loc[df['Source_Type'] == 'Low-carbon'].copy()
country_low_carbon = (
    df_low_carbon.groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values(by='TWh', ascending=False)
)

# 2. Prepare data for the single-level Treemap with a dummy root node
df_treemap = country_low_carbon.copy()
df_treemap['All'] = 'Low-carbon Energy Mix'

fig_treemap = px.treemap(
    df_treemap,
    path=['All', 'Country'],
    values='TWh',
    title="<b>Low-Carbon Energy by Country (Treemap)</b>"
)

fig_treemap.update_traces(
    textinfo="label+value",
    root_color="#E0E0E0"
)
fig_treemap.show()

# 3. Compute insight for the Bar Chart title
top_country = country_low_carbon.iloc[0]['Country']
top_country_val = country_low_carbon.iloc[0]['TWh']

# 4. Build the Horizontal Bar Chart
# Sorting ascending=True displays the highest value at the top in Plotly horizontal bars
fig_bar = px.bar(
    country_low_carbon.sort_values('TWh', ascending=True),
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#0072B2'], # CVD-safe blue
    title=f"<b>Total Low-Carbon TWh by Country</b><br><sup>Insight: {top_country} leads with {top_country_val:,.0f} TWh</sup>"
)

fig_bar.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside'
)

fig_bar.update_layout(
    xaxis_title="Low-Carbon Energy (TWh)",
    yaxis_title="Country",
    margin=dict(t=60, l=25, r=25, b=25),
    font=dict(size=12)
)

fig_bar.show()

Task 2 — Reflection
Which chart tells the ranking story more clearly, and why?

The horizontal bar chart tells the ranking story significantly more clearly.

Direct Linear Comparison: Bar charts map magnitude to a single linear axis (length), allowing the human eye to instantly compare countries and gauge precise relative differences. Treemaps encode values into 2D area rectangles, which are much harder for our visual cortex to accurately parse and rank, especially for values that are close in size.

Effortless Sorting & Labeling: In a horizontal bar chart, items are easily sorted from top to bottom, keeping country labels consistently oriented horizontally and fully legible. In a treemap, text labels frequently clip, rotate, or vanish entirely inside smaller rectangular tiles, making it difficult to read exact rankings or values without hovering.

In [10]:
#TASK 3

import pandas as pd
import plotly.express as px

# 1. Aggregate fossil TWh by Region and Source, then compute Region totals
region_source_fossil = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
region_source_fossil['Region_Total'] = region_source_fossil.groupby('Region')['TWh'].transform('sum')

# 2. Calculate percentage share for each source within its region
region_source_fossil['Share'] = region_source_fossil['TWh'] / region_source_fossil['Region_Total']

# 3. Compute insight: identify the region with the highest coal share
coal_df = region_source_fossil[region_source_fossil['Source'] == 'Coal'].copy()
top_coal_row = coal_df.loc[coal_df['Share'].idxmax()]
top_coal_region = top_coal_row['Region']
top_coal_share = top_coal_row['Share'] * 100

# 4. Sort regions by coal share (ascending order places the highest at the top in Plotly horizontal bars)
sorted_regions = coal_df.sort_values('Share', ascending=True)['Region'].tolist()

# 5. Create labels: only show percentage text for the 'Coal' segments
region_source_fossil['Label'] = region_source_fossil.apply(
    lambda row: f"{row['Share'] * 100:.1f}%" if row['Source'] == 'Coal' else "", 
    axis=1
)

# 6. Define CVD-safe color mapping (Coal matches Task 1 blue, others in distinct greys)
color_map = {
    'Coal': '#0072B2',        # Blue (consistent with Task 1)
    'Natural Gas': '#B0B0B0', # Medium gray
    'Oil': '#E0E0E0'          # Light gray
}

# 7. Build the 100% Stacked Bar Chart
fig = px.bar(
    region_source_fossil,
    x='Share',
    y='Region',
    color='Source',
    orientation='h',
    text='Label',
    color_discrete_map=color_map,
    category_orders={
        'Region': sorted_regions,
        'Source': ['Coal', 'Natural Gas', 'Oil']  # Ensures Coal sits on the baseline
    },
    title=f"<b>Regional Fossil Energy Split (100% Stacked)</b><br><sup>Insight: {top_coal_region} is the most coal-dependent region at {top_coal_share:.1f}%</sup>"
)

fig.update_traces(
    texttemplate='%{text}',
    textposition='inside'
)

fig.update_layout(
    barmode='stack',
    xaxis_title="Share of Fossil Energy Mix",
    yaxis_title="Region",
    xaxis=dict(tickformat='.0%'),
    margin=dict(t=60, l=25, r=25, b=25),
    font=dict(size=12)
)

fig.show()

Task 3 — Reflection
Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.

Task 1 Treemap (Absolute Scale): Answers the question: "Which specific countries and regions consume the highest absolute volume (TWh) of fossil fuels globally, and how do their total market sizes compare to one another?" (It encodes total magnitude via rectangular area).

Task 3 100% Stacked Bar (Proportional Composition): Answers the question: "How do different regions compare purely on internal fuel mix composition, independent of their total energy consumption scale?" (It normalizes every region to a 100% bar, letting you instantly compare relative reliance on coal between a massive energy consumer and a smaller one on an equal footing).